##Ingest Sprints File
1.Read Data from Sprints directory using pyspark dataframe reader
2.Add Ingestion Metdata
    a.Add ingestion timestamp
    b.Add source
3.Write final dataframe to bronze schema

## Step1 .Read data 

In [0]:
%run ../00.common/01.environment_config

In [0]:
%run ../00.common/02.bronze_helpers

In [0]:
source_file = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,DateType,IntegerType,FloatType
sprints_schema = StructType(
    [
        StructField('date',DateType(),True),
        StructField('raceName',StringType(),True),
        StructField('round',IntegerType(),True),
        StructField('season',IntegerType(),True),
        StructField('url',StringType(),True),
        StructField('constructorId',StringType(),True),
        StructField('driverId', StringType(),True),
        StructField('grid',IntegerType(),True),
        StructField('laps',IntegerType(),True),
        StructField('number',IntegerType(),True),
        StructField('points',FloatType(),True),
        StructField('position',IntegerType(),True),
        StructField('positionText',StringType(),True),
        StructField('status',StringType(),True)
    ]
)

In [0]:
# from pyspark.sql.types import StructType,StructField,StringType,FloatType

# circuits_schema = StructType(
#     [
#         StructField('circuitId', StringType(), True),
#         StructField('url', StringType(), True),
#         StructField('circuitName', StringType(), True),
#         StructField('lat', FloatType(), True),
#         StructField('long', FloatType(), True),
#         StructField('locality', StringType(), True),
#         StructField('country', StringType(), True)
#     ]
# )

In [0]:
sprints_df = (
    spark.read
    .format('json')
    .schema(sprints_schema)
    .option('multiLine',True)
    .option('mode','FAILFAST')
    .load(source_file)
)

In [0]:
# display(sprints_df.select(F.col("season")).distinct().orderBy(F.col("season")))

In [0]:
# display(sprints_df)

## Step2 . Add Ingestion Metadata

In [0]:
from pyspark.sql import functions as F
sprints_final_df = add_ingestion_metadata(sprints_df)

##Step3 .Write Data to Delta Table

In [0]:
(
    sprints_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql
-- SELECT season,count(*) as count 
-- FROM formula1.bronze.sprints
-- group by season 
-- ORDER BY season;

In [0]:
# %sql
# SELECT * FROM formula1.bronze.sprints
# WHERE season is NULL;

In [0]:
# display(spark.read.table(table_name))